# PrismaCoder-3B — **End-to-End** on Kaggle: Data Creation → Benchmark → SFT → DPO → GRPO

This notebook runs the **entire PrismaCoder production loop** in a single Kaggle
session (T4 ×2, ~12 h) — no local machine needed:

| Part | What it builds | How |
|---|---|---|
| **A · PrismaForge data** | The Text-to-Prisma training corpus | repo pipelines 01–06: GitHub crawl (mined track) → extract → clean → **synthetic track** (families, personas, reasoning spans) → augment → verification sweep |
| **B · PrismaBench** | The 1,200-task benchmark (T1–T12, 3 levels, canaries, provenance) | seeded mechanical generator + gate review + harness smoke |
| **C · Finalize splits** | train / validation / test + DPO pairs, **contamination firewall** | stages 07–08 (benchmark provenance excluded from training) |
| **D · Training** | **SFT (QLoRA) → DPO → GRPO (online RL) → iterative RL round** | report Table 12.1 hyperparameters, verifier-gated rewards |
| **E · Evaluation** | The trained model scored **on the benchmark built in Part B** | `benchmark/evaluator/evaluate.py` — pass@1, Wilson CI, error taxonomy |
| **F · Ship** | HF Hub push + GGUF export | optional cells |

**Checkpoints everywhere (the point of this notebook)** — every training stage
writes step checkpoints (`adapter + optimizer + scheduler` →
`checkpoints/<stage>_run/checkpoint-N`) and **auto-resumes** from the last one
(`AUTO_RESUME=True`). If Kaggle kills the 12 h session, re-run the notebook with
the previous session's output attached as a Dataset and training continues where
it stopped — nothing is lost. A utility cell copies restored checkpoints into
place automatically.

**Requirements**
- Accelerator: **GPU T4 ×2** (P100 works, slower) · Internet: **ON**
- Kaggle Secrets (Add-ons → Secrets): `HF_TOKEN` (needed for the base model on gated
  quotas + pushing); optional `GITHUB_TOKEN` (enables the mined-track crawl)
- Nothing else — the notebook locates the repo from an attached `PrismaCoder.zip`
  dataset or clones `REPO_URL`.

**Time budget (T4 ×2 defaults below)**: Part A ≈ 25–45 min (crawl-dependent) ·
Part B ≈ 5 min · Part C ≈ 2 min · base eval ≈ 15 min · SFT ≈ 3.5–5 h ·
pair mining ≈ 25 min · DPO ≈ 1.5 h · GRPO ≈ 2 h · RL round ≈ 45 min ·
benchmark eval ≈ 30 min · merges + GGUF ≈ 40 min. **= one long session or two
comfortable ones via checkpoints.** Every part is skippable through `RUN_*` flags.


In [ ]:
# Cell 1 - install the pinned stack (report-tested versions; bump as a set)
%pip install -q -U transformers==4.52.3 trl==0.19.1 peft==0.16.0 \
    datasets==3.6.0 accelerate==1.8.1 bitsandbytes==0.46.1 \
    huggingface-hub==0.33.4 sentencepiece tensorboard matplotlib \
    'PyYAML>=6.0' requests

import importlib
for _pkg in ['transformers', 'trl', 'peft', 'datasets', 'accelerate', 'bitsandbytes']:
    print(_pkg, importlib.import_module(_pkg).__version__)

In [ ]:
# Cell 2 - configuration, stage switches, checkpoint policy, secrets
import os, torch

SEED = 20260830                          # report default seed
BASE_MODEL = 'Qwen/Qwen2.5-Coder-3B-Instruct'
HF_USERNAME = 'prismacoder'              # change to your org/user when pushing
DATASET_REPO = f'{HF_USERNAME}/PrismaForge'
OUTPUT_REPO = f'{HF_USERNAME}/PrismaCoder-3B'
REPO_URL = os.environ.get('PRISMACODER_REPO_URL', 'https://github.com/prismacoder/PrismaCoder.git')

# ---- part switches (skip any part without touching other cells) -----------
RUN_DATA_CREATE = True               # Part A: stages 01-06
CRAWL = True                         # mined-track GitHub crawl (needs GITHUB_TOKEN;
                                     #   without a token it degrades to --offline)
FULL_GATE_DEMO = False               # tiny V1-V5 chain demo (needs Node >= 18.18 + npm)
RUN_BENCH = True                     # Part B: build PrismaBench
RUN_FINALIZE = True                  # Part C: stages 07-08
RUN_TRAIN = True                     # Part D: training stages
RUN_SFT = True
RUN_DPO = True
RUN_GRPO = True                      # the online-RL stage
RUN_ITERATIVE_RL = True              # S4 self-play round on top of GRPO
RUN_BASE_EVAL = True                 # gate pass@1 of the untuned base model
RUN_BENCH_EVAL = True                # Part E: score the final model on PrismaBench
PUSH_TO_HUB = False                  # Part F: push model (+dataset) to the HF Hub
PUSH_DATASET = False                 #   also push the PrismaForge release files
RUN_GGUF = True                      #   export GGUF (llama.cpp convert)
AUTO_RESUME = True                   # continue step-checkpoints across sessions

# ---- Part A scale knobs -----------------------------------------------------
SYNTH_LIMIT = 12000                  # synthetic-track size (full run = 120000; ~66 rec/s)
MAX_TRAIN_SAMPLES = 3000             # SFT cap drawn from the finalized train split (0 = all)
MAX_VAL_SAMPLES = 200

# ---- Part B scale knobs -----------------------------------------------------
BENCH_SCALE = 1.0                    # 1.0 = the full 1,200-task PrismaBench (Table 10.1)
BENCH_EVAL_TASKS = 60                # tasks scored for the final model (deterministic slice)

# ---- SFT (Table 12.1 S1: QLoRA r16/a32, lr 1e-4, cosine 3% warmup) ----------
SFT_EPOCHS = 3
SFT_LR = 1e-4

# ---- DPO (S2: verifier-labeled pairs, beta 0.05, lr 5e-7) -------------------
MAX_DPO_PAIRS = 1500
ONPOLICY_MINING_PROMPTS = 120        # prompts used to mine on-policy pairs
DPO_EPOCHS = 1
DPO_LR = 5e-7
DPO_BETA = 0.05

# ---- GRPO (S3: online RL, verifier reward) ----------------------------------
GRPO_PROMPTS = 120
GRPO_NUM_GENERATIONS = 4
GRPO_MAX_COMPLETION = 640
GRPO_LR = 1e-6
GRPO_BETA = 0.04
GRPO_TEMPERATURE = 0.9

# ---- S4 iterative RL ----------------------------------------------------------
RL2_PROMPTS = 40
RL2_K = 4
RL2_MIN_PAIRS = 8

EVAL_PROMPTS = 40                    # held-out prompts for stage-wise gate pass@1

# ---- checkpoint policy (the resume engine) ----------------------------------
SAVE_STEPS_SFT = 500                 # step interval per stage
SAVE_STEPS_DPO = 400
SAVE_STEPS_GRPO = 25                 # GRPO steps are slow -> save often
SAVE_STEPS_RL2 = 20
SAVE_TOTAL_LIMIT = 3                 # rolling window of step-checkpoints per stage

WORK = '/kaggle/working'
DATA_DIR = f'{WORK}/data'
CKPT = f'{WORK}/checkpoints'
MERGED = f'{WORK}/merged'
STAGE_RESULTS = f'{WORK}/stage_results.json'
FINAL_DIR = None                     # set by the training stages (model to ship)
for _d in (DATA_DIR, CKPT, MERGED):
    os.makedirs(_d, exist_ok=True)

def _secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, '')

HF_TOKEN = _secret('HF_TOKEN')
GITHUB_TOKEN = _secret('GITHUB_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
print('HF token:', 'loaded' if HF_TOKEN else 'not set (public paths only, push skipped)')
print('GITHUB token:', 'loaded (mined track enabled)' if GITHUB_TOKEN
      else 'not set (crawl will be skipped - synthetic track only)')

IS_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
TORCH_DTYPE = torch.bfloat16 if IS_BF16 else torch.float16
print('torch', torch.__version__, '| gpus:', torch.cuda.device_count(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
      '| dtype:', TORCH_DTYPE)

In [ ]:
# Cell 3 - locate / fetch the PrismaCoder repo + a helper for pipeline stages
import glob, os, subprocess, sys, zipfile
from pathlib import Path

def find_repo_root() -> str | None:
    # (a) already-extracted dataset mounted under /kaggle/input
    for pat in ('/kaggle/input/*/prismacoder/__init__.py',
                '/kaggle/input/*/*/prismacoder/__init__.py'):
        hits = sorted(glob.glob(pat))
        if hits:
            return str(Path(hits[0]).parent.parent)
    # (b) PrismaCoder.zip uploaded as a Kaggle dataset -> extract to working
    for z in sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True)):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                if any('prismacoder/__init__.py' in n for n in names):
                    dest = Path(WORK) / 'repo_src'
                    dest.mkdir(parents=True, exist_ok=True)
                    zf.extractall(dest)           # inside the with: zf must stay open
                    # tolerate archives that carry a top-level folder prefix
                    tops = {Path(n).parts[0] for n in names if n.strip()}
                    root_out = dest
                    if len(tops) == 1 and not (dest / 'prismacoder').exists():
                        root_out = dest / tops.pop()
                    print('extracted', z, '->', root_out)
                    return str(root_out)
        except zipfile.BadZipFile:
            continue
    return None

ROOT = find_repo_root()
if ROOT is None:
    print('repo not found under /kaggle/input - cloning', REPO_URL)
    ROOT = str(Path(WORK) / 'PrismaCoder')
    if not Path(ROOT, 'prismacoder', '__init__.py').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, ROOT], check=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from prismacoder import __version__ as pc_version
print('PrismaCoder package', pc_version, '| root:', ROOT)

def run_stage(args, env=None, check=True):
    """Run one repo pipeline stage with live output (exact CLI parity)."""
    args = [str(a) for a in args]
    print('\\$ ' + ' '.join(args), flush=True)
    r = subprocess.run(args, env=env or dict(os.environ), cwd=ROOT)
    if check and r.returncode != 0:
        raise RuntimeError(f'stage failed (exit {r.returncode}): {args}')
    return r

def _nlines(path):
    p = Path(path)
    if not p.exists():
        return None
    return sum(1 for _ in p.open(encoding='utf-8'))

# Part A — PrismaForge data creation (stages 01–06)

Two tracks, merged at verification (report Part 9):

| Stage | Script | What happens |
|---|---|---|
| **01 crawl** | `01_crawl_github.py` | REST-API crawl of seed repos (permissive licenses) for `schema.prisma` + TS call sites. **Needs `GITHUB_TOKEN`**; without one the notebook goes `--offline` and the dataset is 100 % synthetic-track. |
| **02 extract** | `02_extract_corpus.py` | Pull schema records + per-model call-site census from raw snapshots. |
| **03 clean** | `03_clean_corpus.py` | License allowlist, quality gates, dedup — mined "seasoning", not volume. |
| **04 synthetic** | `04_generate_synthetic.py` | The volume track: 5 task families × 12 domain archetypes × personas × difficulty, 10 % carry mechanical plan-spans (reasoning SFT). Fully seeded → reproducible. |
| **05 augment** | `05_augment.py` | Persona paraphrase / entity rename / distractor insertion (20 %) + mined SFT pairs. |
| **06 verify** | `06_verify_all.py` | Gate sweep over every pool. **On Kaggle this runs the static `LightValidator` (V0)** — full V1–V5 needs Node ≥ 18.18 and is a release-time re-run; records keep status `unverified` so that re-run stays visible. |

`SYNTH_LIMIT` scales the same mixture as the report's 120k run (Table 9.1).
12 000 records ≈ 3–4 min on Kaggle CPUs.


In [ ]:
# Cell 5 - stages 01-03: crawl (mined track) -> extract -> clean
if RUN_DATA_CREATE:
    crawl_env = dict(os.environ)
    if GITHUB_TOKEN:
        crawl_env['GITHUB_TOKEN'] = GITHUB_TOKEN
    crawl_args = [] if (CRAWL and GITHUB_TOKEN) else ['--offline']
    if CRAWL and not GITHUB_TOKEN:
        print('GITHUB_TOKEN not set - crawl skipped (--offline); '
              'PrismaForge will be 100% synthetic-track.')
        print('Add the secret (Add-ons -> Secrets -> GITHUB_TOKEN) and re-run to mine repos.')

    run_stage([sys.executable, 'datasets/scripts/01_crawl_github.py',
               '--out', 'datasets', *crawl_args], env=crawl_env, check=False)
    run_stage([sys.executable, 'datasets/scripts/02_extract_corpus.py',
               '--raw', 'datasets/raw', '--out', 'datasets/processed'])
    run_stage([sys.executable, 'datasets/scripts/03_clean_corpus.py',
               '--input', 'datasets/processed/extraction_records.jsonl',
               '--out', 'datasets/processed'])

    print()
    print('extracted records :', _nlines('datasets/processed/extraction_records.jsonl'))
    print('clean records     :', _nlines('datasets/processed/clean_records.jsonl'))
    rep = Path('datasets/processed/cleaning_report.json')
    if rep.exists():
        import json
        cr = json.loads(rep.read_text())
        print('cleaning drops    :', {k: v for k, v in cr.items()
                                       if isinstance(v, (int, float, str))})
else:
    print('RUN_DATA_CREATE=False - skipping Part A stages (wiring will reuse '
          'attached outputs or the hub).')

In [ ]:
# Cell 6 - stages 04-05: synthetic generation (the volume track) + augmentation
import json

if RUN_DATA_CREATE:
    run_stage([sys.executable, 'datasets/scripts/04_generate_synthetic.py',
               '--limit', SYNTH_LIMIT, '--seed', SEED, '--out', 'datasets/synthetic'])
    run_stage([sys.executable, 'datasets/scripts/05_augment.py',
               '--seed', SEED, '--out', 'datasets/synthetic'])

    g = json.loads(Path('datasets/synthetic/generation_report.json').read_text())
    a = json.loads(Path('datasets/synthetic/augment_report.json').read_text())
    print()
    print('synthetic pool :', g['emitted'], 'records (requested', g['requested'], ')')
    fams = g.get('report', {}).get('by_family') or g.get('report', {}).get('families') or {}
    if fams:
        print('by family      :', fams)
    print('augmented      :', a.get('augmented'), '| mined pairs:', a.get('mined_pairs'))
else:
    print('skipped (RUN_DATA_CREATE=False)')

In [ ]:
# Cell 7 - stage 06: verification sweep over every pool (+ optional V1-V5 demo)
import json

if RUN_DATA_CREATE:
    # bulk sweep: static LightValidator (V0) - fast, no Node needed.
    # Records passing V0 keep status 'unverified' -> full-gate re-run stays visible.
    run_stage([sys.executable, 'datasets/scripts/06_verify_all.py', '--light-only'])

    vs = json.loads(Path('datasets/verified/verification_summary.json').read_text())
    print()
    print('mode:', vs['mode'])
    for pool, st in vs['pools'].items():
        print(f"  {pool:<16} records={st['records']:<6} verified={st['verified']:<6} "
              f"failed={st['failed']:<5} unverified={st['unverified']}")

    # optional tiny full-gate demo (V1 prisma validate/generate -> V3 tsc ->
    # V4 db push -> V5 vitest). Needs Node >= 18.18 on the image + npm fetch.
    if FULL_GATE_DEMO:
        from prismacoder.verification.sandbox import node_available
        node, _ver = node_available()
        if node:
            run_stage([sys.executable, 'datasets/scripts/06_verify_all.py',
                       '--input', 'datasets/synthetic/synthetic_pool.jsonl',
                       '--full-gates', '--limit', 25,
                       '--out', 'datasets/verified_sample'])
            print('full-gate sample -> datasets/verified_sample/')
        else:
            print('Node >= 18.18 not available on this image - skipping the V1-V5 demo '
                  '(records stay unverified; re-run 06 with --full-gates on a Node machine).')
else:
    print('skipped (RUN_DATA_CREATE=False)')

# Part B — PrismaBench build (1,200 tasks, T1–T12)

The benchmark is generated by the same seeded mechanical samplers as the report
(`--scale 1.0` = exactly the Table 10.1 composition, checked by assertion):

- **T1–T12 categories** × **L1/L2/L3** × 10 trainable domains (2 withheld for OOD);
- every task's reference must pass the **LightValidator** (adversarial-review stand-in:
  non-green compositions are discarded at build time);
- **canary strings** embedded in every task + a **provenance hash registry** —
  the contamination firewall you will apply to the training split in Part C;
- **public split** (reference withheld) + **private keys** (maintainers only).

The build also writes `benchmark/internal/dev_eval_tasks.jsonl` — an *internal dev
build* that carries references so the **final model can be scored inside this
notebook** (Part E). That file never leaves the session unless you deliberately
upload it; released artifacts stay reference-free.

The harness is smoke-proven right after the build: a dummy submission is scored
through the full evaluator path (pass@1 = 0 by construction — the same baseline
`make smoke` reports locally).


In [ ]:
# Cell 9 - build PrismaBench (public + private + provenance + internal dev refs)
import json
from collections import Counter

if RUN_BENCH:
    from benchmark.generator.generate_benchmark import BenchmarkBuilder
    from prismacoder.constants import BENCH_TASK_COUNTS
    from prismacoder.io_utils import ensure_dir, write_json, write_jsonl

    builder = BenchmarkBuilder(SEED, BENCH_SCALE)
    _captured = {}
    _orig_split = builder._split
    def _capture_split(tasks):
        _captured['tasks'] = tasks          # keep reference-carrying tasks for Part E
        return _orig_split(tasks)
    builder._split = _capture_split

    public_rows, private_rows, provenance = builder.build()
    all_tasks = _captured['tasks']

    ensure_dir('benchmark/public'); ensure_dir('benchmark/private'); ensure_dir('benchmark/internal')
    write_jsonl('benchmark/public/prismabench_public.jsonl', public_rows)
    write_jsonl('benchmark/private/private_keys.jsonl', private_rows)
    write_json('benchmark/public/benchmark_provenance.json', provenance)

    # internal dev build (references visible) - powers the in-notebook final eval
    dev_rows = [
        {
            'bench_id': t['bench_id'], 'category': t['category'], 'level': t['level'],
            'domain': t['domain'], 'instruction': t['instruction'], 'context': t['context'],
            'input': t.get('input', ''), 'reference': t['reference'],
            'artifact_kind': t['artifact_kind'], 'canary': t['canary'],
            'note': 'internal dev build - references visible; do NOT release',
        }
        for t in all_tasks
    ]
    write_jsonl('benchmark/internal/dev_eval_tasks.jsonl', dev_rows)

    by_cat = Counter(t['category'] for t in all_tasks)
    by_level = Counter(t['level'] for t in all_tasks)
    print(f"PrismaBench built: {len(public_rows)} public + {len(private_rows)} private "
          f"= {len(all_tasks)} tasks (scale={BENCH_SCALE})")
    print('categories:', dict(sorted(by_cat.items())))
    print('levels    :', dict(sorted(by_level.items())))
    if BENCH_SCALE >= 0.999:
        assert sum(by_cat.values()) == 1200, f'expected 1200 tasks, got {sum(by_cat.values())}'
        assert dict(by_cat) == BENCH_TASK_COUNTS, 'category counts drifted from Table 10.1'
        print('Table 10.1 composition check passed (1,200 tasks, T1-T12 exact counts)')

    # harness smoke: the dummy submission must score exactly 0 through the
    # full evaluator path (same baseline as `make smoke`)
    run_stage([sys.executable, 'benchmark/evaluator/make_dummy_submission.py',
               '--tasks', 'benchmark/internal/dev_eval_tasks.jsonl',
               '--out', 'benchmark/evaluator/dummy_submission.jsonl'])
    run_stage([sys.executable, 'benchmark/evaluator/evaluate.py',
               '--tasks', 'benchmark/internal/dev_eval_tasks.jsonl',
               '--submission', 'benchmark/evaluator/dummy_submission.jsonl',
               '--out', 'benchmark/leaderboard/smoke_results.json',
               '--model', 'dummy'])
    agg = json.loads(Path('benchmark/leaderboard/smoke_results.json').read_text())['aggregate']
    print('harness smoke: dummy pass@1 =', agg['pass@1']['rate'], '(expected 0.0)')
else:
    print('skipped (RUN_BENCH=False) - Part C will fall back to any committed provenance file.')

# Part C — Finalize the dataset (stages 07–08) with the contamination firewall

Stages 07–08 run **after** the benchmark build on purpose: stage 07 applies the
**contamination firewall** — any training record whose provenance hash appears in
`benchmark/public/benchmark_provenance.json` is excluded from the splits, so the
model is never trained on its own benchmark (report §9.5).

- **stage 07** groups records by *source-schema identity* (not record identity):
  everything derived from one schema lands in exactly one split — no leakage.
  Gate-**failed** records are routed to `negatives.jsonl` (DPO rejected-side pool),
  never imitated by SFT.
- **stage 08** validates every record against the canonical JSON Schema, emits
  `train/validation/test.jsonl` + `dpo_pairs.jsonl` (verifier-labeled gate-differential
  pairs) + `metadata.json` (mixture report) + the dataset card skeleton.


In [ ]:
# Cell 11 - stages 07-08: firewall-aware splits + release set (+ mixture chart)
import json

if RUN_DATA_CREATE and RUN_FINALIZE:
    run_stage([sys.executable, 'datasets/scripts/07_split_dataset.py', '--seed', SEED])
    run_stage([sys.executable, 'datasets/scripts/08_finalize_dataset.py',
               '--splits', 'datasets/splits', '--out', 'datasets', '--tag', 'v1.0-kaggle'])

    sm = json.loads(Path('datasets/splits/split_manifest.json').read_text())
    md_ = json.loads(Path('datasets/metadata.json').read_text())
    print()
    print('split manifest :', {k: sm[k] for k in ('seed', 'databases', 'firewall_entries', 'negatives')})
    print('release set    :', md_['records'], '| dpo_pairs =', md_['dpo_pairs'])
    print('mixture top 8  :', dict(list(md_['mixture_by_category'].items())[:8]))

    import matplotlib.pyplot as plt
    cats = dict(list(md_['mixture_by_category'].items())[:12])
    fig, ax = plt.subplots(figsize=(8, 3.4), constrained_layout=True)
    ax.bar(cats.keys(), cats.values(), color='#6366f1')
    ax.set_title('PrismaForge mixture by category (top 12)')
    ax.set_ylabel('records')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    plt.show()
else:
    print('skipped (RUN_DATA_CREATE/RUN_FINALIZE=False)')

# Part D — Training: SFT → DPO → GRPO → iterative RL (with checkpointing)

**Checkpoint policy (what "save" means here)**

| Artifact | Path | Contents |
|---|---|---|
| Step checkpoints | `checkpoints/<stage>_run/checkpoint-N/` | LoRA adapter **+ optimizer + scheduler + RNG state** (`save_total_limit=3` rolling) |
| Stage adapter | `checkpoints/<stage>_adapter/` | final LoRA adapter for the stage |
| Merged model | `merged/<stage>/` | full fp16 model after `merge_and_unload()` (next stage's base) |
| Progress log | `stage_results.json` | gate pass@1 after every stage |

**Auto-resume**: each trainer calls
`trainer.train(resume_from_checkpoint=get_last_checkpoint(...))` — if step
checkpoints exist (this session, or restored from a previous session's output by
Cell 12b), training continues at the saved step instead of restarting.

**Session-split recipe** (12 h limit): run Parts A–C + SFT in session 1 →
*Save Version* → attach the output as a Dataset in session 2 → re-run from Cell 1
with `RUN_DATA_CREATE=False, RUN_BENCH=False, RUN_FINALIZE=False` → Cell 12b
restores `checkpoints/` and SFT resumes (or start DPO directly — the wiring cell
picks up `datasets/*.jsonl` and `merged/sft` from the attached output).

Hyperparameters = report Table 12.1: QLoRA r16/α32 on all linear projections,
SFT lr 1e-4 (3 epochs, soft-ordering difficulty mixture, completion-only loss),
DPO β=0.05 lr 5e-7 on verifier-labeled pairs, GRPO lr 1e-6 β=0.04 with the
**verifier itself as the reward**, seed 20260830 throughout.


In [ ]:
# Cell 13 - wire the training data: this session's build -> attached outputs -> hub
import glob as _glob
import random
from pathlib import Path

from prismacoder.io_utils import read_jsonl, write_jsonl

def _lines(p) -> int:
    try:
        return sum(1 for _ in Path(p).open(encoding='utf-8'))
    except Exception:
        return 0

def _usable(p, min_records: int = 100) -> bool:
    """A candidate is usable only if it exists and is not the committed
    ~38-record smoke sample that ships inside the repo."""
    return bool(p) and Path(p).exists() and _lines(p) >= min_records

def _pick(name: str, min_records: int = 100) -> str | None:
    """Fresh in-session build wins when it exists; otherwise an attached
    previous-session output; the hub / mini-corpus fallback runs after."""
    in_session = f'datasets/{name}'
    attached = sorted(_glob.glob(f'/kaggle/input/**/datasets/{name}', recursive=True))
    cands = [in_session, *attached] if RUN_DATA_CREATE else [*attached, in_session]
    for c in cands:
        if _usable(c, min_records):
            return c
    return None

train_file = _pick('train.jsonl')
val_file = _pick('validation.jsonl')
pairs_file = _pick('dpo_pairs.jsonl', min_records=1)

if train_file and val_file:
    print('PrismaForge located:')
    print('  train :', train_file, f"({_lines(train_file)} records)")
    print('  val   :', val_file, f"({_lines(val_file)} records)")
else:
    print('no finalized splits found - trying the HF Hub, then the offline mini-corpus')
    from huggingface_hub import hf_hub_download
    train_file = val_file = None
    try:
        train_file = hf_hub_download(DATASET_REPO, 'train.jsonl', repo_type='dataset',
                                     local_dir=str(DATA_DIR))
        val_file = hf_hub_download(DATASET_REPO, 'validation.jsonl', repo_type='dataset',
                                   local_dir=str(DATA_DIR))
        _hub_pairs = hf_hub_download(DATASET_REPO, 'dpo_pairs.jsonl',
                                     repo_type='dataset', local_dir=str(DATA_DIR))
        pairs_file = pairs_file or _hub_pairs
        print('  downloaded from hub:', DATASET_REPO)
    except Exception as e:
        print('  hub unavailable:', type(e).__name__, '-', e)
        print('  building the deterministic mini-corpus (repo synthetic generator)...')
        from prismacoder.synthetic.generator import SyntheticGenerator
        _n = max(MAX_TRAIN_SAMPLES or 4000, 1200) + 250
        _gen = SyntheticGenerator(SEED)
        recs = _gen.generate(_n)
        _gen.attach_reasoning(recs, fraction=0.10)
        _vn = min(MAX_VAL_SAMPLES, max(40, len(recs) // 20))
        write_jsonl(str(Path(DATA_DIR) / 'train.jsonl'), recs[:-_vn])
        write_jsonl(str(Path(DATA_DIR) / 'validation.jsonl'), recs[-_vn:])
        train_file = str(Path(DATA_DIR) / 'train.jsonl')
        val_file = str(Path(DATA_DIR) / 'validation.jsonl')
        pairs_file = None

train_records = read_jsonl(str(train_file))
val_records = read_jsonl(str(val_file))
static_pairs = read_jsonl(str(pairs_file)) if pairs_file else []
print(f'loaded {len(train_records)} train / {len(val_records)} val / '
      f'{len(static_pairs)} static DPO pairs')

if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]

# Deterministic, disjoint prompt pools sliced out of the TRAIN pool
#   [0 : GRPO] -> GRPO RL  |  [GRPO : +MINING] -> on-policy DPO pairs
#   [.. : +RL2] -> iterative RL round.  VALIDATION stays held out for eval.
_rng = random.Random(SEED)
_idx = list(range(len(train_records)))
_rng.shuffle(_idx)
grpo_pool = [train_records[i] for i in _idx[:GRPO_PROMPTS]]
mine_pool = [train_records[i] for i in _idx[GRPO_PROMPTS:GRPO_PROMPTS + ONPOLICY_MINING_PROMPTS]]
rl2_pool = [train_records[i] for i in _idx[GRPO_PROMPTS + ONPOLICY_MINING_PROMPTS:
                                           GRPO_PROMPTS + ONPOLICY_MINING_PROMPTS + RL2_PROMPTS]]
eval_pool = val_records[:EVAL_PROMPTS]
print('pools -> grpo:', len(grpo_pool), '| mining:', len(mine_pool),
      '| rl2:', len(rl2_pool), '| eval (held-out):', len(eval_pool))

In [ ]:
# Cell 14 - 4-bit base model + verifier-gated scorers + checkpoint utilities
import gc, json, re
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.trainer_utils import get_last_checkpoint

from prismacoder.constants import SYSTEM_PROMPT
from prismacoder.verification.verifier import LightValidator

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=TORCH_DTYPE,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    torch_dtype=TORCH_DTYPE,
    attn_implementation='sdpa',      # T4 has no flash-attn wheels
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

LIGHT = LightValidator()             # V0 static gates: PSL parse, fences, symbol grounding
RE_FENCE = re.compile(r'```(?:\w+)?\n(.*?)```', re.S)
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

def extract_artifact(text: str) -> str:
    """Pull the code payload out of a model completion (fenced block preferred)."""
    fences = RE_FENCE.findall(text or '')
    return '\n'.join(fences) if fences else (text or '')

def score_artifact(schema: str, artifact: str) -> tuple[float, str]:
    """Gate-aligned graded score. 1.0 = V0 pass; failure gets shaped partial
    credit so RL gradients still flow; -0.2 for empty output."""
    text = extract_artifact(artifact)
    if not text.strip():
        return -0.2, 'empty'
    verdict = LIGHT.validate(schema, text)
    if verdict.status != 'failed':
        return 1.0, verdict.status
    diags = [d.lower() for d in verdict.diagnostics]
    s = 0.0
    if 'unbalanced code fences' not in diags and '```' in text:
        s += 0.15
    if not any(d.startswith('ungrounded model reference') for d in diags):
        s += 0.15
    if any('ungrounded field' in d for d in diags):
        s += 0.05
    return s, 'failed'

@torch.no_grad()
def generate_completion(messages, *, temperature=0.0, max_new_tokens=640, k=1):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                           add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id)
    if temperature and temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.95,
                          num_return_sequences=k)
    else:
        gen_kwargs.update(do_sample=False, num_return_sequences=k)
    out = model.generate(**inputs, **gen_kwargs)
    return [tokenizer.decode(o[inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            for o in out]

def messages_for(rec: dict) -> list[dict]:
    from training.trainer.data import build_user_message
    return [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_message(rec)}]

def schema_of(rec: dict) -> str:
    return rec['task'].get('context', {}).get('schema', '')

@torch.no_grad()
def gate_pass_rate(records, *, temperature=0.0) -> float:
    """V0 gate pass@1 on held-out records - the stage-progress metric.
    'empty' (degenerate blank completion) counts as a miss, mirroring the
    evaluator's E7 rule."""
    model.eval()
    passed = 0
    for rec in records:
        text = generate_completion(messages_for(rec), temperature=temperature)[0]
        _score, status = score_artifact(schema_of(rec), text)
        passed += status not in ('failed', 'empty')
    return passed / max(1, len(records))

def load_stage_results() -> dict:
    p = Path(STAGE_RESULTS)
    return json.loads(p.read_text()) if p.exists() else {}

def record_stage(name: str, value: float) -> None:
    res = load_stage_results()
    res[name] = round(value, 4)
    Path(STAGE_RESULTS).write_text(json.dumps(res, indent=2))
    print(f'[stage] {name}: gate pass@1 = {value:.3f}')

def free_memory(*objs) -> None:
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def last_ckpt(run_dir: str):
    """Most recent step checkpoint in a run dir, or None (fresh start)."""
    if AUTO_RESUME and os.path.isdir(run_dir):
        try:
            return get_last_checkpoint(run_dir)
        except Exception:
            return None
    return None

def list_ckpts(run_dir: str) -> list[str]:
    return sorted(p for p in glob.glob(f'{run_dir}/checkpoint-*') if os.path.isdir(p))

print('base model ready on', next(model.parameters()).device)

In [ ]:
# Cell 12b - restore step-checkpoints from a previous session's saved output
# (attach the old output via Add Data -> Your Work; this copies it into place so
#  AUTO_RESUME picks it up. In-session runs: this cell is a no-op.)
import glob, shutil

restored = 0
if AUTO_RESUME:
    for run in ('sft_run', 'dpo_run', 'grpo_run', 'rl2_run'):
        for src in glob.glob(f'/kaggle/input/**/checkpoints/{run}/checkpoint-*',
                             recursive=True):
            dst = Path(CKPT) / run / Path(src).name
            if Path(src).is_dir() and not dst.exists():
                shutil.copytree(src, dst)
                restored += 1
    # a previously merged SFT model saves the DPO base download + merge time
    for src in glob.glob('/kaggle/input/**/merged/sft', recursive=True):
        if os.path.isdir(src) and not os.path.isdir(f'{MERGED}/sft'):
            shutil.copytree(src, f'{MERGED}/sft')
            print('restored merged/sft ->', f'{MERGED}/sft')
print('restored step-checkpoints:', restored)

for run in ('sft_run', 'dpo_run', 'grpo_run', 'rl2_run'):
    cks = list_ckpts(f'{CKPT}/{run}')
    if cks:
        print(f'  {run}: resume available at', cks[-1])

In [ ]:
# Cell 14b - baseline: gate pass@1 of the *untuned* base model (the number to beat)
if RUN_TRAIN and RUN_BASE_EVAL:
    record_stage('base', gate_pass_rate(eval_pool))
else:
    print('skipped (RUN_TRAIN/RUN_BASE_EVAL=False)')

## Stage 1 — SFT: imitation of gate-green artifacts (report §12, S1)

Supervised fine-tuning with **completion-only loss** (loss masked to the
assistant span). Hyperparameters follow Table 12.1: QLoRA r=16/α=32 on all
linear projections, lr 1e-4, cosine + 3 % warmup, effective batch 16,
`paged_adamw_8bit`, 3 epochs over the **soft-ordering difficulty mixture**
(epoch 1 oversamples L1, epoch 3 oversamples L3 + repair), the 10 % of records
carrying mechanical plan-spans get them prepended.

Checkpoints: `checkpoints/sft_run/checkpoint-*` every `SAVE_STEPS_SFT` steps
(adapter + optimizer state, limit 3) → final adapter `checkpoints/sft_adapter`
→ merged `merged/sft` (the DPO base).


In [ ]:
# Cell 17a - tokenize PrismaForge into chat examples (completion-only loss)
from datasets import Dataset

from training.trainer.data import SoftOrderingMixer, to_chat_example

def tokenize_record(rec: dict) -> dict:
    ex = to_chat_example(rec, tokenizer, with_reasoning=True)
    return {'input_ids': ex['input_ids'], 'labels': ex['labels']}

if RUN_TRAIN and RUN_SFT:
    _sft_val = ([tokenize_record(r) for r in val_records[:MAX_VAL_SAMPLES]]
                if val_records else [])
    val_ds = Dataset.from_list(_sft_val) if _sft_val else None
    mixer = SoftOrderingMixer(seed=SEED)
    ordered = list(mixer.stream(train_records, SFT_EPOCHS))   # 3 epochs pre-expanded
    train_ds = Dataset.from_list([tokenize_record(r) for r in ordered])
    print(train_ds)
else:
    print('skipped')

In [ ]:
# Cell 17b - QLoRA SFT (plain Trainer, completion-only loss, step-checkpoints)
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments
from torch.nn.utils.rnn import pad_sequence

SFT_RUN = f'{CKPT}/sft_run'

if RUN_TRAIN and RUN_SFT:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )
    model = get_peft_model(model, LoraConfig(
        task_type='CAUSAL_LM', r=16, lora_alpha=32, lora_dropout=0.05,
        bias='none', target_modules=LORA_TARGETS,
    ))
    model.print_trainable_parameters()

    class LossMaskCollator:
        """Pad pre-tokenized examples; labels padded with -100 (ignored)."""
        def __init__(self, pad_id: int):
            self.pad_id = pad_id

        def __call__(self, feats):
            ids = [torch.tensor(f['input_ids']) for f in feats]
            lbs = [torch.tensor(f['labels']) for f in feats]
            ids = pad_sequence(ids, batch_first=True, padding_value=self.pad_id)
            lbs = pad_sequence(lbs, batch_first=True, padding_value=-100)
            return {'input_ids': ids, 'labels': lbs,
                    'attention_mask': (ids != self.pad_id).long()}

    args = TrainingArguments(
        output_dir=SFT_RUN,
        run_name='prismacoder-3b-sft-kaggle',
        seed=SEED,
        num_train_epochs=1,                       # the mixer already repeats 3 epochs
        learning_rate=SFT_LR,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,            # effective batch 16
        bf16=IS_BF16,
        fp16=not IS_BF16,                         # T4: fp16 + GradScaler
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit',
        max_grad_norm=1.0,
        weight_decay=0.01,
        logging_steps=20,
        eval_strategy='steps' if val_ds is not None else 'no',
        eval_steps=250,
        # ---- checkpoint saving (the resume engine) ----
        save_strategy='steps',
        save_steps=SAVE_STEPS_SFT,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to=['tensorboard'],
        remove_unused_columns=False,
        dataloader_num_workers=2,
    )

    sft_trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=LossMaskCollator(tokenizer.pad_token_id),
    )
    _ck = last_ckpt(SFT_RUN)
    if _ck:
        print('RESUMING SFT from', _ck)
    sft_trainer.train(resume_from_checkpoint=_ck)      # None -> fresh start
    sft_trainer.save_model(f'{CKPT}/sft_adapter')
    tokenizer.save_pretrained(f'{CKPT}/sft_adapter')
    print('SFT adapter ->', f'{CKPT}/sft_adapter')
    print('step checkpoints kept:', list_ckpts(SFT_RUN))
    free_memory(sft_trainer)
else:
    print('RUN_SFT=False - skipping (re-run later; resume picks up saved checkpoints)')

In [ ]:
# Cell 17c - merge the SFT adapter -> merged/sft (the DPO base), then score it
import shutil, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

SFT_MERGED = f'{MERGED}/sft'

if RUN_TRAIN and RUN_SFT:
    free_memory(model)
    base_cpu = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=TORCH_DTYPE, device_map='cpu', trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base_cpu, f'{CKPT}/sft_adapter')
    merged = peft_model.merge_and_unload()
    merged.save_pretrained(SFT_MERGED, safe_serialization=True)
    tokenizer.save_pretrained(SFT_MERGED)
    free_memory(base_cpu, peft_model, merged)
    print('merged SFT ->', SFT_MERGED)

    # reload the merged SFT model in 4-bit for everything that follows
    model = AutoModelForCausalLM.from_pretrained(
        SFT_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = False
    record_stage('sft', gate_pass_rate(eval_pool))
elif os.path.isdir(f'{MERGED}/sft'):
    SFT_MERGED = f'{MERGED}/sft'
    print('RUN_SFT=False - reusing restored merged model:', SFT_MERGED)
else:
    SFT_MERGED = None
    print('skipped (RUN_SFT=False)')

## Stage 2 — DPO: preference optimization on verifier-labeled pairs (report §12, S2)

DPO needs *pairs* (chosen ≻ rejected). PrismaForge labels them with the same
verifier used everywhere else (**gate-differential basis**: chosen = gate-green
reference, rejected = the strongest gate-failing mutant — stage 08's
`dpo_pairs.jsonl`). This stage also mines **on-policy pairs**: the SFT model
samples two completions per prompt, the V0 gate scores both, and the
differential becomes a fresh pair — the standard bridge from offline DPO to
online RL.

- Static pairs: `datasets/dpo_pairs.jsonl` (built in Part C) or hub fallback.
- On-policy pairs: `ONPOLICY_MINING_PROMPTS` × 2 samples, kept only when the
  gate verdicts actually differ.
- Hyperparameters (Table 12.1): β=0.05, lr 5e-7, 1 epoch, effective batch 16,
  reference = the SFT model (adapter-disable path — no second model in memory).

Checkpoints: `checkpoints/dpo_run/checkpoint-*` every `SAVE_STEPS_DPO` steps →
adapter `checkpoints/dpo_adapter` → merged `merged/dpo` (the GRPO base).


In [ ]:
# Cell 18a - build the DPO pair set (static gate-differential + on-policy mining)
import random
from pathlib import Path

from prismacoder.io_utils import write_jsonl

DPO_MERGED = f'{MERGED}/dpo'
final_pairs = [p for p in static_pairs if p.get('chosen') and p.get('rejected')]

if RUN_TRAIN and RUN_DPO and mine_pool:
    rng = random.Random(SEED + 1)
    mined = 0
    for rec in mine_pool:
        outs = generate_completion(messages_for(rec), temperature=0.8,
                                   max_new_tokens=512, k=2)
        scored = [(*score_artifact(schema_of(rec), t), t) for t in outs]
        scored.sort(key=lambda x: x[0], reverse=True)
        (hi_s, hi_st, hi_t), (lo_s, lo_st, lo_t) = scored[0], scored[-1]
        # keep the pair only when the gate actually separates the two samples
        if hi_s > lo_s and (hi_st != 'failed' or lo_st == 'failed' or hi_s - lo_s >= 0.2):
            final_pairs.append({
                'id': f"{rec['id']}-onpolicy",
                'prompt': rec['task']['instruction'],
                'context_schema': schema_of(rec),
                'chosen': extract_artifact(hi_t),
                'rejected': extract_artifact(lo_t),
                'basis': f'on-policy gate differential ({hi_st} vs {lo_st})',
            })
            mined += 1
    print(f'on-policy mining: {mined}/{len(mine_pool)} prompts produced separable pairs')

rng = random.Random(SEED + 2)
rng.shuffle(final_pairs)
final_pairs = final_pairs[:MAX_DPO_PAIRS]
DPO_PAIRS_FILE = str(Path(DATA_DIR) / 'dpo_pairs_final.jsonl')
write_jsonl(DPO_PAIRS_FILE, final_pairs)
print('DPO pair set:', len(final_pairs), '->', DPO_PAIRS_FILE)

In [ ]:
# Cell 18b - DPO training (TRL DPOTrainer, ref = adapter-disabled SFT model)
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer

from training.trainer.data import to_dpo_example

DPO_RUN = f'{CKPT}/dpo_run'

if RUN_TRAIN and RUN_DPO:
    free_memory(model)
    dpo_base_dir = SFT_MERGED or BASE_MODEL
    model = AutoModelForCausalLM.from_pretrained(
        dpo_base_dir, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )

    pair_rows = read_jsonl(DPO_PAIRS_FILE)
    dpo_ds = Dataset.from_list([to_dpo_example(p, tokenizer) for p in pair_rows])
    print(dpo_ds)

    dpo_args = DPOConfig(
        output_dir=DPO_RUN,
        run_name='prismacoder-3b-dpo-kaggle',
        seed=SEED,
        num_train_epochs=DPO_EPOCHS,
        learning_rate=DPO_LR,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,            # effective batch 16
        bf16=IS_BF16,
        fp16=not IS_BF16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit',
        max_grad_norm=1.0,
        beta=DPO_BETA,                            # 0.05 per report Table 12.1
        loss_type='sigmoid',
        max_prompt_length=1536,
        max_completion_length=1024,
        logging_steps=10,
        # ---- checkpoint saving ----
        save_strategy='steps',
        save_steps=SAVE_STEPS_DPO,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to=['tensorboard'],
    )

    dpo_trainer = DPOTrainer(
        model=model,
        args=dpo_args,
        train_dataset=dpo_ds,
        processing_class=tokenizer,
        peft_config=LoraConfig(
            task_type='CAUSAL_LM', r=16, lora_alpha=32, lora_dropout=0.05,
            bias='none', target_modules=LORA_TARGETS,
        ),
    )
    _ck = last_ckpt(DPO_RUN)
    if _ck:
        print('RESUMING DPO from', _ck)
    dpo_trainer.train(resume_from_checkpoint=_ck)
    dpo_trainer.save_model(f'{CKPT}/dpo_adapter')
    tokenizer.save_pretrained(f'{CKPT}/dpo_adapter')
    print('DPO adapter ->', f'{CKPT}/dpo_adapter')
    print('step checkpoints kept:', list_ckpts(DPO_RUN))
    free_memory(dpo_trainer)
else:
    print('RUN_DPO=False - skipping')

In [ ]:
# Cell 18c - merge the DPO adapter -> merged/dpo (the GRPO base), then score it
import os, shutil
from peft import PeftModel
from transformers import AutoModelForCausalLM

if RUN_TRAIN and RUN_DPO:
    free_memory(model)
    dpo_base_dir = SFT_MERGED or BASE_MODEL
    base_cpu = AutoModelForCausalLM.from_pretrained(
        dpo_base_dir, torch_dtype=TORCH_DTYPE, device_map='cpu', trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base_cpu, f'{CKPT}/dpo_adapter')
    merged = peft_model.merge_and_unload()
    merged.save_pretrained(DPO_MERGED, safe_serialization=True)
    tokenizer.save_pretrained(DPO_MERGED)
    free_memory(base_cpu, peft_model, merged)
    print('merged DPO ->', DPO_MERGED)

    if SFT_MERGED and os.path.isdir(SFT_MERGED) and DPO_MERGED != SFT_MERGED:
        shutil.rmtree(SFT_MERGED)          # merged/sft (~6 GB) no longer needed
        print('freed disk:', SFT_MERGED)

    model = AutoModelForCausalLM.from_pretrained(
        DPO_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = False
    record_stage('dpo', gate_pass_rate(eval_pool))
elif os.path.isdir(f'{MERGED}/dpo'):
    DPO_MERGED = f'{MERGED}/dpo'
    print('RUN_DPO=False - reusing restored merged model:', DPO_MERGED)
else:
    DPO_MERGED = SFT_MERGED or BASE_MODEL
    print('skipped (RUN_DPO=False) - GRPO will start from', DPO_MERGED)

## Stage 3 — GRPO: online reinforcement learning with the verifier as reward (report §12, S3)

This is the **RL stage**. GRPO (Group Relative Policy Optimization — the
PPO-family algorithm behind DeepSeek-R1) removes the learned critic: for every
prompt the policy samples a *group* of `num_generations` completions, each is
scored by the **same Prisma verifier that filtered the dataset and labeled the
DPO pairs**, advantages are normalized within the group, and the policy gradient
is PPO-clipped with a KL term (`beta`) against the frozen reference (the DPO
model, adapters disabled).

**Reward (gate-aligned, graded)** — `+1.0` V0 pass; failure gets shaped partial
credit (fences, grounding); `-0.2` empty output. On a Node machine the same
function swaps in `Verifier.verify()` for the full V1–V5 chain — the reward *is*
the gate chain, which is the project's core idea.

Checkpoints: `checkpoints/grpo_run/checkpoint-*` every `SAVE_STEPS_GRPO` steps
(= 25 — GRPO steps are generation-heavy and slow, so save often) → adapter
`checkpoints/grpo_adapter` → merged `merged/grpo`.


In [ ]:
# Cell 19a - the verifier reward function + the GRPO prompt dataset
from datasets import Dataset

def prisma_verifier_reward(completions, schema_psl=None, **kwargs):
    """GRPO reward = the Prisma verifier. Extra dataset columns (here
    `schema_psl`) are forwarded by GRPOTrainer as kwargs lists."""
    schemas = schema_psl or [''] * len(completions)
    rewards = []
    for completion, schema in zip(completions, schemas):
        text = completion[0]['content'] if isinstance(completion, list) else completion
        score, _status = score_artifact(schema, text)
        rewards.append(score)
    return rewards

if RUN_TRAIN and RUN_GRPO:
    grpo_rows = []
    for rec in grpo_pool:
        if not schema_of(rec).strip():
            continue
        grpo_rows.append({
            'prompt': messages_for(rec),          # conversational prompt (TRL applies template)
            'schema_psl': schema_of(rec),         # extra column -> reward kwarg
            'task_id': rec['id'],
        })
    grpo_ds = Dataset.from_list(grpo_rows)
    print(grpo_ds)
    print('example reward check:',
          prisma_verifier_reward([grpo_rows[0]['prompt'][1]['content']],
                                 [grpo_rows[0]['schema_psl']]))
else:
    print('skipped')

In [ ]:
# Cell 19b - GRPO training (TRL GRPOTrainer + QLoRA, verifier reward)
import torch
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

GRPO_RUN = f'{CKPT}/grpo_run'
GRPO_MERGED = f'{MERGED}/grpo'

if RUN_TRAIN and RUN_GRPO:
    free_memory(model)
    model = AutoModelForCausalLM.from_pretrained(
        DPO_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = True            # generation-heavy stage: KV cache ON
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )

    grpo_args = GRPOConfig(
        output_dir=GRPO_RUN,
        run_name='prismacoder-3b-grpo-kaggle',
        seed=SEED,
        num_train_epochs=1,
        learning_rate=GRPO_LR,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,        # 2*1*4 = 8 completions/step = 2 groups of 4
        num_generations=GRPO_NUM_GENERATIONS, # group size for relative advantages
        max_prompt_length=1536,
        max_completion_length=GRPO_MAX_COMPLETION,
        temperature=GRPO_TEMPERATURE,
        beta=GRPO_BETA,                       # KL vs adapter-disabled DPO policy
        use_vllm=False,                       # transformers generation (T4-safe)
        bf16=IS_BF16,
        fp16=not IS_BF16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        optim='paged_adamw_8bit',
        max_grad_norm=1.0,
        logging_steps=5,
        # ---- checkpoint saving (frequent: GRPO steps are expensive) ----
        save_strategy='steps',
        save_steps=SAVE_STEPS_GRPO,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to=['tensorboard'],
    )

    grpo_trainer = GRPOTrainer(
        model=model,
        reward_funcs=[prisma_verifier_reward],
        args=grpo_args,
        train_dataset=grpo_ds,
        processing_class=tokenizer,
        peft_config=LoraConfig(
            task_type='CAUSAL_LM', r=16, lora_alpha=32, lora_dropout=0.05,
            bias='none', target_modules=LORA_TARGETS,
        ),
    )
    _ck = last_ckpt(GRPO_RUN)
    if _ck:
        print('RESUMING GRPO from', _ck)
    grpo_trainer.train(resume_from_checkpoint=_ck)
    grpo_trainer.save_model(f'{CKPT}/grpo_adapter')
    tokenizer.save_pretrained(f'{CKPT}/grpo_adapter')
    print('GRPO adapter ->', f'{CKPT}/grpo_adapter')
    print('step checkpoints kept:', list_ckpts(GRPO_RUN))
    free_memory(grpo_trainer)
else:
    print('RUN_GRPO=False - skipping')

In [ ]:
# Cell 19c - merge the GRPO adapter -> merged/grpo, then score it
import os, shutil
from peft import PeftModel
from transformers import AutoModelForCausalLM

if RUN_TRAIN and RUN_GRPO:
    free_memory(model)
    base_cpu = AutoModelForCausalLM.from_pretrained(
        DPO_MERGED, torch_dtype=TORCH_DTYPE, device_map='cpu', trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base_cpu, f'{CKPT}/grpo_adapter')
    merged = peft_model.merge_and_unload()
    merged.save_pretrained(GRPO_MERGED, safe_serialization=True)
    tokenizer.save_pretrained(GRPO_MERGED)
    free_memory(base_cpu, peft_model, merged)
    print('merged GRPO ->', GRPO_MERGED)

    if RUN_DPO and os.path.isdir(DPO_MERGED) and DPO_MERGED != BASE_MODEL \
            and DPO_MERGED != GRPO_MERGED:
        shutil.rmtree(DPO_MERGED)
        print('freed disk:', DPO_MERGED)

    model = AutoModelForCausalLM.from_pretrained(
        GRPO_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
        attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
    )
    model.config.use_cache = False
    record_stage('grpo', gate_pass_rate(eval_pool))
elif os.path.isdir(f'{MERGED}/grpo'):
    GRPO_MERGED = f'{MERGED}/grpo'
    print('RUN_GRPO=False - reusing restored merged model:', GRPO_MERGED)
else:
    GRPO_MERGED = DPO_MERGED
    print('skipped (RUN_GRPO=False)')

## Stage 4 — Iterative RL: the continual self-play loop (report §12.4)

One GRPO pass is a single RL iteration; production runs the loop **repeatedly**:
sample *k* completions per fresh prompt from the current policy → the verifier
picks winners/losers → the differential distills into a short DPO round → merge
→ repeat. This cell runs one extra iteration (RAFT-style best-of-*k*) on
prompts disjoint from everything seen so far. Duplicate it (with a fresh prompt
slice) for more rounds.

Checkpoints: `checkpoints/rl2_run/checkpoint-*` every `SAVE_STEPS_RL2` steps →
merged `merged/rl2` = **the final model**.


In [ ]:
# Cell 20 - iterative RL round: best-of-k self-play pairs -> short DPO -> merge
import os, shutil
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer

RL2_RUN = f'{CKPT}/rl2_run'
RL2_MERGED = f'{MERGED}/rl2'
FINAL_DIR = GRPO_MERGED

if RUN_TRAIN and RUN_ITERATIVE_RL and rl2_pool:
    # 4a. sample k completions per fresh prompt from the current policy
    selfplay_pairs = []
    for rec in rl2_pool:
        schema = schema_of(rec)
        if not schema.strip():
            continue
        outs = generate_completion(messages_for(rec), temperature=1.0,
                                   max_new_tokens=512, k=RL2_K)
        scored = sorted(((score_artifact(schema, t)[0], t) for t in outs),
                        key=lambda x: x[0], reverse=True)
        best_s, best_t = scored[0]
        worst_s, worst_t = scored[-1]
        if best_s >= 0.8 and best_s > worst_s:      # winner must clear the gate
            selfplay_pairs.append({
                'prompt': rec['task']['instruction'],
                'context_schema': schema,
                'chosen': extract_artifact(best_t),
                'rejected': extract_artifact(worst_t),
                'basis': f'self-play best-of-{RL2_K} ({best_s:.2f} vs {worst_s:.2f})',
            })
    print('self-play pairs:', len(selfplay_pairs))

    if len(selfplay_pairs) >= RL2_MIN_PAIRS:
        # 4b. short DPO round on the current policy (checkpoints + resume)
        free_memory(model)
        model = AutoModelForCausalLM.from_pretrained(
            GRPO_MERGED, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
            attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
        )
        model.config.use_cache = False
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={'use_reentrant': False},
        )
        rl2_args = DPOConfig(
            output_dir=RL2_RUN, run_name='prismacoder-3b-rl-round2-kaggle',
            seed=SEED, num_train_epochs=1, learning_rate=DPO_LR,
            lr_scheduler_type='cosine', warmup_ratio=0.03,
            per_device_train_batch_size=2, gradient_accumulation_steps=8,
            bf16=IS_BF16, fp16=not IS_BF16,
            gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
            optim='paged_adamw_8bit', max_grad_norm=1.0, beta=DPO_BETA,
            loss_type='sigmoid', max_prompt_length=1536, max_completion_length=1024,
            logging_steps=5,
            save_strategy='steps', save_steps=SAVE_STEPS_RL2,
            save_total_limit=SAVE_TOTAL_LIMIT, report_to=['tensorboard'],
        )
        rl2_trainer = DPOTrainer(
            model=model, args=rl2_args,
            train_dataset=Dataset.from_list(
                [to_dpo_example(p, tokenizer) for p in selfplay_pairs]),
            processing_class=tokenizer,
            peft_config=LoraConfig(task_type='CAUSAL_LM', r=16, lora_alpha=32,
                                   lora_dropout=0.05, bias='none',
                                   target_modules=LORA_TARGETS),
        )
        _ck = last_ckpt(RL2_RUN)
        if _ck:
            print('RESUMING RL round 2 from', _ck)
        rl2_trainer.train(resume_from_checkpoint=_ck)
        rl2_trainer.save_model(f'{CKPT}/rl2_adapter')
        print('RL2 adapter ->', f'{CKPT}/rl2_adapter')
        free_memory(rl2_trainer)

        # 4c. merge -> final model
        free_memory(model)
        base_cpu = AutoModelForCausalLM.from_pretrained(
            GRPO_MERGED, torch_dtype=TORCH_DTYPE, device_map='cpu', trust_remote_code=True)
        peft_model = PeftModel.from_pretrained(base_cpu, f'{CKPT}/rl2_adapter')
        merged = peft_model.merge_and_unload()
        merged.save_pretrained(RL2_MERGED, safe_serialization=True)
        tokenizer.save_pretrained(RL2_MERGED)
        free_memory(base_cpu, peft_model, merged)

        if os.path.isdir(GRPO_MERGED) and GRPO_MERGED != RL2_MERGED:
            shutil.rmtree(GRPO_MERGED)
            print('freed disk:', GRPO_MERGED)

        FINAL_DIR = RL2_MERGED
        model = AutoModelForCausalLM.from_pretrained(
            FINAL_DIR, quantization_config=bnb, torch_dtype=TORCH_DTYPE,
            attn_implementation='sdpa', device_map='auto', trust_remote_code=True,
        )
        model.config.use_cache = False
        record_stage('rl_round2', gate_pass_rate(eval_pool))
    else:
        print(f'only {len(selfplay_pairs)} separable pairs (< {RL2_MIN_PAIRS}) '
              f'- skipping the RL round; final model stays at', FINAL_DIR)
else:
    print('RUN_ITERATIVE_RL=False or empty pool - skipping')

# Part E — Score the trained model on PrismaBench (the benchmark built in Part B)

The final model generates submissions for a deterministic slice of
`benchmark/internal/dev_eval_tasks.jsonl` (references embedded — internal dev
build only), and the repo's own evaluator scores them:

- **pass@1** (primary; V0 static gates on Kaggle, full V1–V5 on a Node machine),
- **exact match / CodeBLEU / tree similarity**, **composite score**
  (0.20·compile + 0.60·execute + 0.10·style + 0.10·efficiency),
- **failure taxonomy E1–E10** + per-category breakdown — the same leaderboard
  pipeline as `make eval`.


In [ ]:
# Cell 21 - generate submissions on PrismaBench and score them with evaluate.py
import json, time

BENCH_RESULTS = None
if RUN_TRAIN and RUN_BENCH_EVAL and Path('benchmark/internal/dev_eval_tasks.jsonl').exists():
    from prismacoder.io_utils import write_jsonl

    dev_tasks = read_jsonl('benchmark/internal/dev_eval_tasks.jsonl')[:BENCH_EVAL_TASKS]
    print(f'generating {len(dev_tasks)} benchmark submissions '
          f'(greedy, max_new_tokens=768)...')
    rows = []
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    model.eval()
    for i, t in enumerate(dev_tasks):
        rec = {'task': {'instruction': t['instruction'],
                        'context': t['context'], 'input': t.get('input', '')}}
        t0 = time.time()
        text = generate_completion(messages_for(rec), temperature=0.0,
                                   max_new_tokens=768)[0]
        lat = int((time.time() - t0) * 1000)
        mem = int(torch.cuda.max_memory_allocated() / 2**20) if torch.cuda.is_available() else 0
        rows.append({'bench_id': t['bench_id'], 'output': text,
                     'latency_ms': lat, 'memory_mb': mem})
        if (i + 1) % 10 == 0:
            print(f'  {i + 1}/{len(dev_tasks)} generated ({lat} ms)')

    SUB = f'{WORK}/final_submission.jsonl'
    write_jsonl(SUB, rows)
    run_stage([sys.executable, 'benchmark/evaluator/evaluate.py',
               '--tasks', 'benchmark/internal/dev_eval_tasks.jsonl',
               '--submission', SUB,
               '--out', 'benchmark/leaderboard/prismacoder3b_final.json',
               '--model', 'prismacoder-3b-kaggle',
               '--max-tasks', str(len(dev_tasks))])
    run_stage([sys.executable, 'benchmark/evaluator/leaderboard.py',
               '--results', 'benchmark/leaderboard',
               '--out', 'benchmark/leaderboard/LEADERBOARD.md'])

    res = json.loads(Path('benchmark/leaderboard/prismacoder3b_final.json').read_text())
    BENCH_RESULTS = res['aggregate']
    print()
    print('PrismaBench pass@1:', BENCH_RESULTS['pass@1']['rate'],
          ' Wilson95:', BENCH_RESULTS['pass@1'].get('wilson95'))
    for k in ('exact_match', 'code_bleu', 'tree_similarity'):
        if k in BENCH_RESULTS:
            print(f'  {k:<18}', BENCH_RESULTS[k])
    per_cat = {}
    for r in res['rows']:
        c = per_cat.setdefault(r['category'], [0, 0])
        c[0] += int(r['passed']); c[1] += 1
    print('per-category pass@1:')
    for cat, (p, n) in sorted(per_cat.items()):
        print(f'  {cat:<24} {p}/{n} ({p / n:.2f})')
    errs = {}
    for r in res['rows']:
        if r.get('error_class'):
            errs[r['error_class']] = errs.get(r['error_class'], 0) + 1
    print('error taxonomy:', errs or '(none failing)')
elif not RUN_BENCH_EVAL:
    print('skipped (RUN_BENCH_EVAL=False)')
else:
    print('skipped (no internal dev tasks - build the benchmark in Part B first)')

In [ ]:
# Cell 22 - stage-wise progress: gate pass@1 table + chart
import matplotlib.pyplot as plt

results = load_stage_results()
order = [(k, results[k]) for k in ('base', 'sft', 'dpo', 'grpo', 'rl_round2') if k in results]
print('| Stage | V0 gate pass@1 (held-out) |')
print('|---|---|')
for k, v in order:
    print(f'| {k} | {v:.3f} |')
if BENCH_RESULTS is not None:
    print(f"| PrismaBench pass@1 ({BENCH_EVAL_TASKS} tasks) | "
          f"{BENCH_RESULTS['pass@1']['rate']:.3f} |")

if len(order) >= 2:
    labels = [k for k, _ in order]
    values = [v for _, v in order]
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    bars = ax.bar(labels, values,
                  color=['#94a3b8', '#8b5cf6', '#6366f1', '#4f46e5', '#312e81'][:len(order)])
    ax.axhline(results.get('base', 0), color='#94a3b8', linestyle='--', linewidth=1)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('V0 gate pass@1 (held-out)')
    ax.set_title('PrismaCoder-3B: base -> SFT -> DPO -> GRPO -> RL round 2')
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f'{v:.2f}',
                ha='center', va='bottom', fontsize=9)
    out_png = f'{WORK}/stage_progress.png'
    fig.savefig(out_png, dpi=150)
    json.dump(dict(order), open(f'{WORK}/stage_results_final.json', 'w'), indent=2)
    print('chart ->', out_png)
    plt.show()

# Part F — Ship the artifacts

Everything under `/kaggle/working` persists in the notebook **Output** tab once
you *Save Version*:

| Artifact | Path |
|---|---|
| Final merged model | `merged/rl2` (or `merged/grpo`) |
| Stage adapters | `checkpoints/*_adapter/` |
| **Step checkpoints (resume)** | `checkpoints/*_run/checkpoint-*/` |
| PrismaForge release set | `datasets/train.jsonl`, `validation.jsonl`, `test.jsonl`, `dpo_pairs.jsonl`, `metadata.json` |
| PrismaBench | `benchmark/public/` (+ private keys, maintainers only) |
| Leaderboard results | `benchmark/leaderboard/` |
| GGUF | `PrismaCoder-3B-q8_0.gguf` |

The two cells below (optional) push the model + dataset to the HF Hub and export
GGUF for llama.cpp/Ollama.


In [ ]:
# Cell 23 - push the final model (+ optionally PrismaForge) to the HF Hub
from huggingface_hub import HfApi

if PUSH_TO_HUB and HF_TOKEN and FINAL_DIR:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(OUTPUT_REPO, repo_type='model', private=True, exist_ok=True)
    api.upload_large_folder(repo_id=OUTPUT_REPO, repo_type='model', folder_path=FINAL_DIR)
    api.upload_file(path_or_fileobj=STAGE_RESULTS, path_in_repo='stage_results.json',
                    repo_id=OUTPUT_REPO, repo_type='model')
    card = (
        '---\n'
        'license: apache-2.0\n'
        f'base_model: {BASE_MODEL}\n'
        'tags: [prisma, orm, text-to-code, qlora, dpo, grpo]\n'
        '---\n\n'
        '# PrismaCoder-3B\n\n'
        'Qwen2.5-Coder-3B-Instruct trained end to end: PrismaForge data creation -> '
        'PrismaBench -> SFT -> DPO (verifier-labeled pairs) -> GRPO (verifier reward, '
        'online RL) -> iterative RL round. Built entirely in the Kaggle end-to-end '
        'notebook.\n\n'
        f'Stage-wise V0 gate pass@1: {json.dumps(load_stage_results())}\n'
    )
    api.upload_file(path_or_fileobj=card.encode(), path_in_repo='README.md',
                    repo_id=OUTPUT_REPO, repo_type='model')
    print('pushed model:', OUTPUT_REPO)

    if PUSH_DATASET:
        api.create_repo(DATASET_REPO, repo_type='dataset', private=True, exist_ok=True)
        for fname in ('train.jsonl', 'validation.jsonl', 'test.jsonl',
                      'dpo_pairs.jsonl', 'metadata.json'):
            p = Path('datasets') / fname
            if p.exists():
                api.upload_file(path_or_fileobj=str(p), path_in_repo=fname,
                                repo_id=DATASET_REPO, repo_type='dataset')
        print('pushed dataset:', DATASET_REPO)
elif not PUSH_TO_HUB:
    print('PUSH_TO_HUB=False - model kept at', FINAL_DIR)
else:
    print('need HF_TOKEN and a trained model (FINAL_DIR) to push')

In [ ]:
# Cell 24 - export GGUF (llama.cpp convert -> f16 -> q8_0)
import os, subprocess, sys

if FINAL_DIR:
    LC_DIR = '/kaggle/working/llama.cpp'
    if not os.path.exists(LC_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/ggml-org/llama.cpp', LC_DIR], check=True)
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                        f'{LC_DIR}/requirements.txt'], check=False)
        f16 = f'{WORK}/PrismaCoder-3B-f16.gguf'
        q8 = f'{WORK}/PrismaCoder-3B-q8_0.gguf'
        subprocess.run([sys.executable, f'{LC_DIR}/convert_hf_to_gguf.py', FINAL_DIR,
                        '--outfile', f16, '--outtype', 'f16'], check=True)
        if os.path.exists(f'{LC_DIR}/build/bin/llama-quantize'):
            subprocess.run([f'{LC_DIR}/build/bin/llama-quantize', f16, q8, 'q8_0'],
                           check=True)
        else:
            q8 = f16  # quantize binary not built - keep the f16 export
        if os.path.exists(q8) and q8 != f16 and os.path.exists(f16):
            os.remove(f16)                # stay inside the /kaggle/working quota
        print('GGUF ->', q8)
    except Exception as e:
        print('GGUF export failed (non-fatal):', type(e).__name__, '-', e)
else:
    print('no trained model this session - skipping GGUF export')

## What you just built — and what to do next

One Kaggle session produced the **full PrismaCoder production loop**:
`PrismaForge (crawl + synthetic + verify + splits) → PrismaBench (1,200 tasks,
canaries, provenance firewall) → SFT → DPO → GRPO (online RL, verifier reward)
→ iterative RL round → benchmark-scored final model`, with **step checkpoints
for every stage** and auto-resume across sessions.

**Resume / session-split cheat-sheet**
1. End of a session: *Save Version* (output persists under `/kaggle/working`).
2. New session: attach that output (Add Data → Your Work) and set
   `RUN_DATA_CREATE=False, RUN_BENCH=False, RUN_FINALIZE=False` (data arrives
   from the attached output), keep the stages you still need at `True`.
3. Cell 12b copies restored checkpoints into place; each trainer resumes from
   the last step checkpoint automatically.

**Next steps**
- Full-gate evaluation on a Node ≥ 18 machine:
  `python benchmark/evaluator/evaluate.py --tasks benchmark/public/prismabench_public.jsonl --submission <generations> --model your-name --full-gates`
  (V0 passes here are `unverified` until the V1–V5 chain re-runs them).
- Scale to the report's 120k run: `SYNTH_LIMIT=120000`, `MAX_TRAIN_SAMPLES=0`,
  `GRPO_PROMPTS=1000` — same notebook, longer sessions, same checkpoints.
- Compare models pairwise: `evaluation/compare_models.py` (McNemar + bootstrap + Holm).

**Troubleshooting**

| Symptom | Fix |
|---|---|
| CUDA OOM in SFT/DPO | `per_device_train_batch_size=1`, `gradient_accumulation_steps=16` |
| GRPO slow | lower `GRPO_PROMPTS` (e.g. 80) or `GRPO_MAX_COMPLETION` (e.g. 448); generation dominates |
| `num_generations` divisibility error | keep `per_device_train_batch_size × grad_accum` a multiple of 4 (defaults 2×4=8 ✓) |
| fp16 loss spike on T4 | `SFT_EPOCHS=2`, keep `max_grad_norm=1.0`; T4 has no bf16 |
| Session killed at hour 12 | that's what the checkpoints are for — follow the cheat-sheet above |
| Crawl yields few records | expected on Kaggle without `GITHUB_TOKEN`; the synthetic track carries the volume (report two-track design) |
| Merges OOM (13 GB RAM sessions) | use T4 ×2 (29 GB RAM) or merge with `torch_dtype=torch.float16` |

**Packages**: transformers 4.52.3 · trl 0.19.1 · peft 0.16.0 · bitsandbytes 0.46.1 —
pinned in Cell 1; bump as a set, not individually.
